# Explorer une question

Ce cahier montre, pour une question, **ce que la recherche trouve et pourquoi**,
et permet de comparer plusieurs réglages (nombre de passages `k`, seuil de
similarité) sans rien payer : le LLM n'est appelé que si on le demande
explicitement (dernière cellule).

Prérequis : `uv sync --group notebooks`, l'index construit (`uv run rag-index`),
puis choisir le noyau `.venv` en haut à droite du cahier.

Lancer les cellules une par une avec **Maj + Entrée**.

In [ ]:
import os
from pathlib import Path

# Travailler depuis la racine du dépôt (data/, .env), où que VS Code ouvre le cahier.
if Path.cwd().name == "notebooks":
    os.chdir("..")

import matplotlib.pyplot as plt
import pandas as pd

from rag_rgpd.explain.render import format_trace
from rag_rgpd.explain.trace import explain_question, write_trace
from rag_rgpd.retrieval.embeddings import SentenceTransformerEmbedder
from rag_rgpd.retrieval.index import DEFAULT_K, DEFAULT_MIN_SIMILARITY, Searcher, read_meta

## 1. Charger l'index et BGE-M3

Environ une minute, une seule fois : les cellules suivantes réutilisent `searcher`.

In [ ]:
index_dir = Path("data/index")
searcher = Searcher(index_dir, SentenceTransformerEmbedder(read_meta(index_dir).embedding_model))
searcher.meta

## 2. Les passages retrouvés pour une question

On regarde large (`k=12` plus 8 de réserve) pour voir aussi ce qui se trouve
juste après la coupure habituelle (`k=8`).

- **similarité** : proximité de sens avec la question (cosinus, 1 = identique) ;
  c'est elle que la barrière 1 compare au seuil ;
- **RRF** : score de fusion des deux recherches, qui fixe l'ordre ;
- **rang sens / rang mots** : place du passage dans chacune des deux recherches
  avant la fusion. Un passage bien classé par le sens mais absent par les mots
  (ou l'inverse) explique souvent un classement surprenant.

In [ ]:
QUESTION = "Puis-je demander l'effacement de mes données ?"

trace, _ = explain_question(QUESTION, searcher, None, k=12, extra=8)
passages = pd.DataFrame([
    {"rang": p.rank, "document": p.document_id, "endroit": p.label,
     "similarité": p.similarity, "RRF": p.score,
     "rang sens": p.meaning_rank, "rang mots": p.words_rank}
    for p in trace.passages
])
passages

In [ ]:
SEUIL = DEFAULT_MIN_SIMILARITY
K = DEFAULT_K

fig, ax = plt.subplots(figsize=(10, 4))
colors = ["tab:blue" if s >= SEUIL else "tab:gray" for s in passages["similarité"]]
ax.bar(passages["rang"], passages["similarité"], color=colors)
ax.axhline(SEUIL, color="tab:red", linestyle="--", label=f"seuil {SEUIL}")
ax.axvline(K + 0.5, color="black", linestyle=":", label=f"coupure k={K}")
ax.set_xlabel("rang après fusion")
ax.set_ylabel("similarité cosinus")
ax.set_title(QUESTION)
ax.legend()
plt.show()

## 3. Comparer plusieurs valeurs de `k`

Le classement hybride dépend un peu du nombre de candidats demandés : on
regarde ce qui entre et sort quand `k` change.

In [ ]:
rows = []
for k in (4, 8, 12, 16):
    t, _ = explain_question(QUESTION, searcher, None, k=k, extra=0)
    rows.append({"k": k, "meilleure similarité": t.best_similarity,
                 "passages envoyés": [p.label for p in t.sent_passages]})
pd.DataFrame(rows)

## 4. Le même seuil pour d'autres questions

Où tombent des questions hors sujet par rapport au seuil ? Utile pour régler
la barrière 1 avant l'étape 5.

In [ ]:
QUESTIONS = [
    QUESTION,
    "Qu'est-ce qu'une donnée sensible ?",
    "article 17",
    "Quelle est la capitale du Pérou ?",
    "Comment réussir une pâte à crêpes ?",
]
best = pd.DataFrame([
    {"question": q, "meilleure similarité": explain_question(q, searcher, None)[0].best_similarity}
    for q in QUESTIONS
])
ax = best.plot.barh(x="question", y="meilleure similarité", legend=False, figsize=(8, 3))
ax.axvline(SEUIL, color="tab:red", linestyle="--")
plt.show()
best

## 5. Avec le LLM (payant)

Environ 0,001 $ par question avec Haiku. Le cache local évite de payer deux
fois exactement la même requête. Passer `APPELER_LE_LLM` à `True`.

In [ ]:
APPELER_LE_LLM = False

if APPELER_LE_LLM:
    from dotenv import load_dotenv

    from rag_rgpd.answering.cache import ResponseCache
    from rag_rgpd.answering.generator import ClaudeGenerator

    load_dotenv(".env", override=False, encoding="utf-8")
    trace, answer = explain_question(QUESTION, searcher, ClaudeGenerator(cache=ResponseCache()))
    write_trace(trace)  # relisible ensuite avec `uv run rag-trace`
    print(format_trace(trace))